# Create Non Coding Data Set
Andrew E. Davidson
aedaivds@ucsc.edu 02/10/25

Copyright (c) 2020-2023, Regents of the University of California All rights reserved. https://polyformproject.org/licenses/noncommercial/1.0.0

ref: /private/groups/kimlab/data/elife/README.md


1. Get the bioytype from an old Create run  
    * Load the Elife 20241107/create/annotated_norm_counts.csv
    * extra the gene_id column and the biotype column

2. ~~Load the tempus/illumina/20241220/results/data/normalized_counts.csv~~
    * this does not have the biotype columnm

3. load the elife count data
   * /private/groups/kimlab/data/elife/elife_all_norm_counts_2023-05-18.csv

4. join on gene_id

5. create a 'Lung Cancer', 'Healthy donor' only data, drop all coding genes

**data files** 
18 rows are missing after join  
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_all_norm_counts_2023-05-18.csv 

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

outDir:
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data


In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
#print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/lungCancer/../../../../deconvolutionAnalysis/python:/private/home/aedavid

In [3]:
from intraExtraRNA.elifeUtilities import loadMetaData

In [4]:
%%time
tempusPath = "/private/groups/kimlab/data/tempus/illumina/20241107/create/annotated_norm_counts.csv"
tempusCountsDF= pd.read_csv( tempusPath, index_col='gene_id' )

CPU times: user 234 ms, sys: 59.6 ms, total: 294 ms
Wall time: 684 ms


In [5]:
print( tempusCountsDF.shape )
tempusCountsDF.iloc[0:5, 0:4]

(76539, 37)


,gene_name,gene_biotype,SLDK3_T1_100_S1_L007,SLDK3_T1_100K_S2_L007
gene_id,,,,
(A)n,(A)n,Microsatellite,0.545009,1.987324
(AAA)n,(AAA)n,Microsatellite,0.000000,0.000000
(AAAAAAC)n,(AAAAAAC)n,Microsatellite,0.000000,0.000000
(AAAAAAG)n,(AAAAAAG)n,Microsatellite,0.000000,0.000000
(AAAAAAT)n,(AAAAAAT)n,Microsatellite,0.000000,0.000000


In [6]:
%%time
elifePath = "/private/groups/kimlab/data/elife/elife_all_norm_counts_2023-05-18.csv"
elifeCountsDF = pd.read_csv(elifePath, index_col='gene')

CPU times: user 1.83 s, sys: 207 ms, total: 2.04 s
Wall time: 2.88 s


In [7]:
print( elifeCountsDF.shape )
elifeCountsDF.iloc[0:5, 0:5]

(76555, 224)


,SRR14506659,SRR14506660,SRR14506661,SRR14506662,SRR14506663
gene,,,,,
(A)n,201.672053,110.450773,3722.776395,1394.605651,2843.47373
(AAA)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAC)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAG)n,0.000000,0.000000,0.000000,0.000000,0.00000
(AAAAAAT)n,0.000000,0.000000,0.000000,0.000000,0.00000


In [8]:
print(f'the number of rows is different elifeCountsDF.shape[0] - tempusCountsDF.shape[0] = {elifeCountsDF.shape[0] - tempusCountsDF.shape[0]}' )

the number of rows is different elifeCountsDF.shape[0] - tempusCountsDF.shape[0] = 16


In [9]:
%%time

elifeWithBiotypeDF = pd.merge( 
                            tempusCountsDF.loc[:, "gene_biotype"], 
                            elifeCountsDF, 
                            how='inner',
                            left_index=True,
                            right_index=True,
                            # left_on='gene_id', 
                            # right_on='gene', 
                            suffixes=('_t', '_e') )

elifeWithBiomarkerDF.index.name = "gene"

NameError: name 'elifeWithBiomarkerDF' is not defined

In [10]:
print(elifeWithBiotypeDF.shape)

(76537, 225)


In [11]:
elifeWithBiotypeDF.iloc[0:5, 0:5]

,gene_biotype,SRR14506659,SRR14506660,SRR14506661,SRR14506662
(A)n,Microsatellite,201.672053,110.450773,3722.776395,1394.605651
(AAA)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAC)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAG)n,Microsatellite,0.000000,0.000000,0.000000,0.000000
(AAAAAAT)n,Microsatellite,0.000000,0.000000,0.000000,0.000000


In [12]:
savePath = f'{dataOutDir}/annotated_elife_all_norm_counts_2023-05-18.csv'
elifeWithBiotypeDF.to_csv( savePath )
print( f'saved:\n{savePath}')

saved:
/private/groups/kimlab/aedavids/elife/createNonCodingDataSet.out/data/annotated_elife_all_norm_counts_2023-05-18.csv


# Create a data of set of lung samples, drop all coding genes

In [13]:
elifeMetaDF = loadMetaData()

In [14]:
elifeMetaDF['diagnosis'].unique()

array(['Esophagus Cancer', 'Lung Cancer', 'Liver Cancer',
       'Stomach Cancer', 'Colorectal Cancer', 'Healthy donor'],
      dtype=object)

In [15]:
selelectLungRows = elifeMetaDF['diagnosis'].isin( ['Lung Cancer', 'Healthy donor'] )
lungSampleIds = elifeMetaDF.loc[ selelectLungRows, :]
lungSampleIds

,sample_id,diagnosis
31,SRR14506690,Lung Cancer
32,SRR14506691,Lung Cancer
33,SRR14506692,Lung Cancer
34,SRR14506693,Lung Cancer
35,SRR14506694,Lung Cancer
...,...,...
219,SRR14506884,Healthy donor
220,SRR14506885,Healthy donor
221,SRR14506886,Healthy donor
222,SRR14506887,Healthy donor


In [16]:
lungCols = ['gene_biotype'] + list( lungSampleIds['sample_id'].values )
elifeLungDF = elifeWithBiotypeDF.loc[:, lungCols ]
print(elifeLungDF.shape)
elifeLungDF.head()

(76537, 79)


,gene_biotype,SRR14506690,SRR14506691,SRR14506692,SRR14506693,SRR14506694,SRR14506695,SRR14506696,SRR14506697,SRR14506698,...,SRR14506879,SRR14506880,SRR14506881,SRR14506882,SRR14506883,SRR14506884,SRR14506885,SRR14506886,SRR14506887,SRR14506888
(A)n,Microsatellite,148.057125,121.600515,551.413204,45.545968,86.068765,127.856929,42.539172,524.589036,132.83713,...,139412.051757,815421.544118,11389.259,35304.023371,39039.547964,37425.679475,509.276433,35.048632,105.06569,66.661012
(AAA)n,Microsatellite,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
(AAAAAAC)n,Microsatellite,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
(AAAAAAG)n,Microsatellite,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
(AAAAAAT)n,Microsatellite,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
